## Scrape BLS data without using an API

In [ ]:
url= "https://data.bls.gov/timeseries/CES0000000001"

base_url="https://data.bls.gov/timeseries/"
series_id="CES0000000001"
request_parameters = f"?years_option=specific_years&from_year=2021&delimiter=comma&output_type=column&periods_option=all_periods&output_view=net_1mth&output_format=html&include_graphs=False"

full_url=base_url+series_id+request_parameters
full_url

In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get(full_url)
doc = BeautifulSoup(response.text, 'html.parser')
response.raise_for_status()

In [ ]:

 # get to the rows
table = doc.find("table", {"id": "table0"})

if not table:
	print(f"No table found for {series_id}")

In [ ]:

tbody = table.find("tbody")
if not tbody:
	print(f"No rows found for {series_id}")

In [ ]:
trs=tbody.find_all("tr")

In [ ]:
rows=[]
for tr in trs:
    cells = tr.find_all(["th", "td"], recursive=False)

    raw_value = cells[3].get_text(strip=True)
    is_preliminary = raw_value.endswith("(P)")
    clean_value = raw_value.replace("(P)", "").replace(",", "").strip()
    
    rows.append({
        "series_id": cells[0].get_text(strip=True),
        "year": int(cells[1].get_text(strip=True)),
        "period": cells[2].get_text(strip=True),
        "jobs_added": int(clean_value),
        "is_prelim": is_preliminary,
    })
import pandas as pd
df=pd.DataFrame(rows)

df.info()

In [ ]:
df['month']=df['year'].astype(str)+"-"+df['period'].str.replace("M", "")
df['time_stamp']=pd.to_datetime(df['month'])


df=df.drop_duplicates()

In [ ]:

rows = []
for tr in tbody.find_all("tr"):
    cells = tr.find_all(["th", "td"])
    if len(cells) != 4:
        continue

    raw_value = cells[3].get_text(strip=True)
    is_preliminary = raw_value.endswith("(P)")
    clean_value = raw_value.replace("(P)", "").replace(",", "").strip()

    rows.append({
        "series_id": cells[0].get_text(strip=True),
        "year": int(cells[1].get_text(strip=True)),
        "period": cells[2].get_text(strip=True),
        "jobs_added": int(clean_value),
        "is_prelim": is_preliminary,
    })

import pandas as pd
df = pd.DataFrame(rows)
# df=clean_cols(df)
df['month']=df['year'].astype(str)+"-"+df['period'].str.replace("M", "")
df['time_stamp']=pd.to_datetime(df['month'])

df=df.drop_duplicates()

In [ ]:
import matplotlib.pyplot as plt

df.plot(x='time_stamp', y='jobs_added', kind='bar', figsize=(10, 6))

In [ ]:
import numpy as np

conditions = [
    df["time_stamp"] <= "2025-01-01",
    df["time_stamp"] > "2025-01-01",
]

choices =['biden', 'trump_second']

df["president"] = np.select(conditions, choices, default='unknown')

df.president.value_counts()


In [ ]:
## lambda function:

df['lambda_prez']= df['time_stamp'].apply(lambda x: 'biden' if x <= pd.to_datetime('2025-01-01') else 'trump_second')


## a regular function

def get_president(time_stamp):
    if time_stamp <= pd.to_datetime('2025-01-01'):
        return 'biden'
    else:
        return 'trump_second'

df['regular_prez'] = df['time_stamp'].apply(get_president)



In [ ]:

## pd.cut 

bins = [pd.to_datetime('2020-01-01'), pd.to_datetime('2025-02-01'), pd.to_datetime('2030-01-01')]
labels=['biden', 'trump_second']

df['pd_cut_prez'] = pd.cut(df['time_stamp'], bins=bins, labels=labels, right=False)

In [ ]:
df[[ 'president', 'lambda_prez', 'regular_prez', 'pd_cut_prez']].value_counts()

In [ ]:
df['added_or_lost']=df.jobs_added.apply(lambda x: 'added' if x > 0 else 'lost')

In [ ]:
df.head()

### Inflation data

In [ ]:
# API: https://data.bls.gov/timeseries/CUUR0000SA0?years_option=specific_years&to_year=2025&from_year=2015&delimiter=comma&output_type=column&periods_option=all_periods&output_view=pct_12mths&output_format=html&original_include_graphs=False&original_output_view=pct_12mths&include_graphs=False

base_url="https://data.bls.gov/timeseries/"
series_id="CUUR0000SA0"

prefix='CUUR0000'

period_option=['pct_12mths', 'pct_1mth', 'date']


major_items_dictionary={
	"All items":"SA0",
	"Food": "SAF1",
	# "Energy":"SA0E",
	# "Electricity":"SEHF01",
	# "Gasoline":"SETB01",
	# "Apparel":"SAA",
	# "New Vehicles":"SETA01",
	# "Used cars and trucks": "SETA02",
	# 'Shelter':"SAH1",
    # "Food at home": "SAF11",
	# "Food away from home": "SEFV"
}


full_url= base_url + series_id + request_parameters

full_url

In [ ]:
base_url="https://data.bls.gov/timeseries/"
request_parameters = f"?years_option=specific_years&from_year=2019&delimiter=comma&output_type=column&periods_option=all_periods&output_format=html&include_graphs=False"

def fetch_bls_data(series_id):

    url= base_url+prefix+series_id+request_parameters
    response = requests.get(url)
    response.raise_for_status()
    doc = BeautifulSoup(response.text, 'html.parser')

     # get to the rows
    table = doc.find("table", {"id": "table0"})
    if not table:
        print(f"No table found for {series_id}")

    tbody = table.find("tbody")
    if not tbody:
        print(f"No rows found for {series_id}")
    

    data = []

    for row in tbody.find_all("tr"):
        cells = row.find_all(["th","td"])
        cells = [cell.get_text(strip=True) for cell in cells]
        if cells:
            data.append(cells)

    df = pd.DataFrame(data, columns=["series_id", "year", "month", "value"])

    return df

In [ ]:
dfs=[]

for category, series_id in major_items_dictionary.items():
    df = fetch_bls_data(series_id)
    df['category'] = category
    print(df.shape)
    dfs.append(df)
    # Do something with the fetched data, e.g., save it to a file or upload it

master_df = pd.concat(dfs, ignore_index=True)

In [ ]:
    # Clean up
remove_months = ['S01', 'S02', 'M13']
master_df = master_df[~master_df.month.isin(remove_months)].reset_index(drop=True)
# master_df = master_df[master_df['value']!="(X)"].reset_index(drop=True)
# master_df = master_df[master_df['value']!="-(X)"].reset_index(drop=True)
master_df['value'] = master_df['value'].replace({"(X)": np.nan, "-(X)": np.nan})
master_df["date"] = master_df.year + "-" + master_df.month.str.replace("M", "")
master_df["value"] = master_df["value"].astype(float)

In [ ]:
pvtd_df=master_df.pivot(index='date', columns='category', values='value').reset_index()

In [ ]:
pvtd_df.head()

In [ ]:
plt.style.use('fivethirtyeight')

pvtd_df.plot(x='date', y=['All items', 'Food'], kind='line', figsize=(10, 6))